# 🎯 Capa 4: Orquestación

Este notebook integra todas las capas en un flujo completo.

**Componentes:**
- Clase `EcoMarketBot` que encapsula toda la lógica
- Manejo de sesiones de conversación
- Logging de interacciones
- Preparación para la interfaz Streamlit

In [1]:
import os
import json
import sqlite3
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../.env')
print('✅ Entorno cargado')

✅ Entorno cargado


## 1. Clase EcoMarketBot

Encapsula toda la funcionalidad del chatbot en una sola clase reutilizable.

In [2]:
class EcoMarketBot:
    """
    Chatbot del supermercado EcoMarket.
    Integra OpenAI LLM con funciones de consulta a SQLite.
    """
    
    MODEL = 'gpt-4o-mini'
    MAX_RETRIES = 2
    
    SYSTEM_PROMPT = """
Eres EcoBot, el asistente virtual del supermercado EcoMarket. Eres amable, eficiente y servicial.

CAPACIDADES:
1. Consultar disponibilidad de productos (por nombre o categoría)
2. Informar sobre el estado de pedidos (el cliente se identifica con su email)
3. Mostrar promociones vigentes y recomendar según lo que consulte el cliente
4. Gestionar devoluciones (registrar nuevas, consultar estado)
5. Derivar a un agente humano cuando no puedas resolver algo

REGLAS:
- Siempre responde en español
- Sé conciso pero amable
- Si un producto tiene promoción activa, menciónala automáticamente
- Para pedidos y devoluciones, siempre pide el email del cliente si no lo tienes
- Si no puedes resolver algo, ofrece derivar a un agente humano
- Para derivar a humano necesitas: email, descripción del problema, y opcionalmente número de pedido
- No inventes información. Si no encuentras datos, dilo claramente
- Usa emojis moderadamente para ser más amigable
"""
    
    def __init__(self, db_path: str = '../data/ecomarket.db'):
        self.db_path = db_path
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.historial = []
        self.tools = self._definir_tools()
        self.funciones = self._mapear_funciones()
    
    def _get_conn(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn
    
    def resetear_conversacion(self):
        """Reinicia el historial de conversación."""
        self.historial = []
    
    def responder(self, mensaje_usuario: str) -> str:
        """
        Procesa un mensaje y devuelve la respuesta del bot.
        Mantiene el historial para contexto.
        """
        self.historial.append({'role': 'user', 'content': mensaje_usuario})
        mensajes = [{'role': 'system', 'content': self.SYSTEM_PROMPT}] + self.historial
        
        try:
            response = self.client.chat.completions.create(
                model=self.MODEL,
                messages=mensajes,
                tools=self.tools,
                tool_choice='auto',
                max_tokens=1024,
                temperature=0.3
            )
        except Exception as e:
            # Si falla tool calling, reintentar sin tools
            response = self.client.chat.completions.create(
                model=self.MODEL,
                messages=mensajes,
                max_tokens=1024,
                temperature=0.3
            )
            respuesta = response.choices[0].message.content
            self.historial.append({'role': 'assistant', 'content': respuesta})
            return respuesta
        
        message = response.choices[0].message
        
        # Ciclo de function calling
        while message.tool_calls:
            self.historial.append({
                'role': 'assistant',
                'content': message.content,
                'tool_calls': [
                    {
                        'id': tc.id,
                        'type': 'function',
                        'function': {
                            'name': tc.function.name,
                            'arguments': tc.function.arguments
                        }
                    } for tc in message.tool_calls
                ]
            })
            
            for tool_call in message.tool_calls:
                nombre_fn = tool_call.function.name
                args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
                resultado = self._ejecutar_funcion(nombre_fn, args)
                
                self.historial.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': resultado
                })
            
            mensajes = [{'role': 'system', 'content': self.SYSTEM_PROMPT}] + self.historial
            try:
                response = self.client.chat.completions.create(
                    model=self.MODEL,
                    messages=mensajes,
                    tools=self.tools,
                    tool_choice='auto',
                    max_tokens=1024,
                    temperature=0.3
                )
                message = response.choices[0].message
            except Exception:
                response = self.client.chat.completions.create(
                    model=self.MODEL,
                    messages=mensajes,
                    max_tokens=1024,
                    temperature=0.3
                )
                message = response.choices[0].message
                break
        
        respuesta = message.content
        self.historial.append({'role': 'assistant', 'content': respuesta})
        return respuesta
    
    def _ejecutar_funcion(self, nombre: str, args: dict) -> str:
        if nombre not in self.funciones:
            return json.dumps({'error': f'Función {nombre} no encontrada'})
        if not args:
            args = {}
        resultado = self.funciones[nombre](**args)
        return json.dumps(resultado, ensure_ascii=False, default=str)
    
    # ═══════════════════════════════════════════
    # FUNCIONES DE CONSULTA (integradas)
    # ═══════════════════════════════════════════
    
    def _buscar_producto(self, nombre: str) -> list:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute(
            'SELECT id, nombre, categoria, precio, stock, descripcion, unidad '
            'FROM productos WHERE LOWER(nombre) LIKE LOWER(?)',
            (f'%{nombre}%',)
        )
        resultados = [dict(row) for row in cursor.fetchall()]
        
        # Agregar info de promoción si existe
        hoy = datetime.now().strftime('%Y-%m-%d')
        for prod in resultados:
            cursor.execute(
                'SELECT descripcion, descuento_porcentaje FROM promociones '
                'WHERE producto_id = ? AND activa = 1 AND fecha_inicio <= ? AND fecha_fin >= ?',
                (prod['id'], hoy, hoy)
            )
            promo = cursor.fetchone()
            if promo:
                prod['promocion'] = dict(promo)
        
        conn.close()
        return resultados
    
    def _buscar_por_categoria(self, categoria: str) -> list:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute(
            'SELECT id, nombre, categoria, precio, stock, unidad '
            'FROM productos WHERE LOWER(categoria) LIKE LOWER(?) ORDER BY nombre',
            (f'%{categoria}%',)
        )
        resultados = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return resultados
    
    def _consultar_pedidos(self, email: str) -> list:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute(
            'SELECT id, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada '
            'FROM pedidos WHERE LOWER(email_cliente) = LOWER(?) ORDER BY fecha_pedido DESC',
            (email,)
        )
        resultados = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return resultados
    
    def _obtener_detalle_pedido(self, pedido_id: int) -> dict:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM pedidos WHERE id = ?', (pedido_id,))
        pedido = cursor.fetchone()
        if not pedido:
            conn.close()
            return {'error': 'Pedido no encontrado'}
        cursor.execute(
            'SELECT p.nombre, dp.cantidad, dp.precio_unitario, '
            '(dp.cantidad * dp.precio_unitario) as subtotal '
            'FROM detalle_pedido dp JOIN productos p ON p.id = dp.producto_id '
            'WHERE dp.pedido_id = ?',
            (pedido_id,)
        )
        items = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return {'pedido': dict(pedido), 'items': items}
    
    def _obtener_promociones(self) -> list:
        conn = self._get_conn()
        cursor = conn.cursor()
        hoy = datetime.now().strftime('%Y-%m-%d')
        cursor.execute(
            'SELECT pr.id, p.nombre as producto, pr.descripcion, '
            'pr.descuento_porcentaje, pr.fecha_fin, p.precio as precio_original, '
            'ROUND(p.precio * (1 - pr.descuento_porcentaje/100), 2) as precio_con_descuento '
            'FROM promociones pr JOIN productos p ON p.id = pr.producto_id '
            'WHERE pr.activa = 1 AND pr.fecha_inicio <= ? AND pr.fecha_fin >= ? '
            'ORDER BY pr.descuento_porcentaje DESC',
            (hoy, hoy)
        )
        resultados = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return resultados
    
    def _registrar_devolucion(self, pedido_id: int, email: str, motivo: str) -> dict:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute(
            'SELECT id, estado FROM pedidos WHERE id = ? AND LOWER(email_cliente) = LOWER(?)',
            (pedido_id, email)
        )
        pedido = cursor.fetchone()
        if not pedido:
            conn.close()
            return {'error': 'Pedido no encontrado o no pertenece a este email'}
        if dict(pedido)['estado'] not in ['entregado', 'enviado']:
            conn.close()
            return {'error': f'No se puede devolver un pedido con estado: {dict(pedido)["estado"]}'}
        fecha = datetime.now().strftime('%Y-%m-%d')
        cursor.execute(
            'INSERT INTO devoluciones (pedido_id, email_cliente, motivo, estado, fecha_solicitud) '
            'VALUES (?, ?, ?, "pendiente", ?)',
            (pedido_id, email, motivo, fecha)
        )
        devolucion_id = cursor.lastrowid
        conn.commit()
        conn.close()
        return {'mensaje': 'Devolución registrada', 'devolucion_id': devolucion_id, 'estado': 'pendiente'}
    
    def _consultar_devoluciones(self, email: str) -> list:
        conn = self._get_conn()
        cursor = conn.cursor()
        cursor.execute(
            'SELECT id, pedido_id, motivo, estado, fecha_solicitud, comentarios '
            'FROM devoluciones WHERE LOWER(email_cliente) = LOWER(?) '
            'ORDER BY fecha_solicitud DESC',
            (email,)
        )
        resultados = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return resultados
    
    def _crear_ticket_soporte(self, email: str, asunto: str, descripcion: str, pedido_id: int = None) -> dict:
        conn = self._get_conn()
        cursor = conn.cursor()
        fecha = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        cursor.execute(
            'INSERT INTO tickets_soporte (email_cliente, asunto, descripcion, estado, fecha_creacion, pedido_id) '
            'VALUES (?, ?, ?, "abierto", ?, ?)',
            (email, asunto, descripcion, fecha, pedido_id)
        )
        ticket_id = cursor.lastrowid
        conn.commit()
        conn.close()
        return {'mensaje': 'Ticket creado. Un agente se pondrá en contacto pronto.', 'ticket_id': ticket_id}
    
    def _mapear_funciones(self):
        return {
            'buscar_producto': self._buscar_producto,
            'buscar_por_categoria': self._buscar_por_categoria,
            'consultar_pedidos': self._consultar_pedidos,
            'obtener_detalle_pedido': self._obtener_detalle_pedido,
            'obtener_promociones': self._obtener_promociones,
            'registrar_devolucion': self._registrar_devolucion,
            'consultar_devoluciones': self._consultar_devoluciones,
            'crear_ticket_soporte': self._crear_ticket_soporte,
        }
    
    def _definir_tools(self):
        # (misma definición de TOOLS del notebook 03)
        return [
            {"type": "function", "function": {"name": "buscar_producto", "description": "Busca productos por nombre en el catálogo.", "parameters": {"type": "object", "properties": {"nombre": {"type": "string", "description": "Nombre o parte del nombre del producto"}}, "required": ["nombre"]}}},
            {"type": "function", "function": {"name": "buscar_por_categoria", "description": "Lista productos de una categoría. Categorías: Frutas y Verduras, Lácteos, Carnes, Panadería, Bebidas, Limpieza y Hogar, Despensa", "parameters": {"type": "object", "properties": {"categoria": {"type": "string", "description": "Categoría de productos"}}, "required": ["categoria"]}}},
            {"type": "function", "function": {"name": "consultar_pedidos", "description": "Consulta pedidos de un cliente por email.", "parameters": {"type": "object", "properties": {"email": {"type": "string", "description": "Email del cliente"}}, "required": ["email"]}}},
            {"type": "function", "function": {"name": "obtener_detalle_pedido", "description": "Detalle completo de un pedido (productos, cantidades, precios).", "parameters": {"type": "object", "properties": {"pedido_id": {"type": "integer", "description": "ID del pedido"}}, "required": ["pedido_id"]}}},
            {"type": "function", "function": {"name": "obtener_promociones", "description": "Muestra todas las promociones vigentes.", "parameters": {"type": "object", "properties": {}, "required": []}}},
            {"type": "function", "function": {"name": "registrar_devolucion", "description": "Registra solicitud de devolución.", "parameters": {"type": "object", "properties": {"pedido_id": {"type": "integer", "description": "ID del pedido"}, "email": {"type": "string", "description": "Email del cliente"}, "motivo": {"type": "string", "description": "Razón de la devolución"}}, "required": ["pedido_id", "email", "motivo"]}}},
            {"type": "function", "function": {"name": "consultar_devoluciones", "description": "Consulta devoluciones de un cliente.", "parameters": {"type": "object", "properties": {"email": {"type": "string", "description": "Email del cliente"}}, "required": ["email"]}}},
            {"type": "function", "function": {"name": "crear_ticket_soporte", "description": "Crea ticket para derivar a agente humano.", "parameters": {"type": "object", "properties": {"email": {"type": "string", "description": "Email del cliente"}, "asunto": {"type": "string", "description": "Asunto del problema"}, "descripcion": {"type": "string", "description": "Descripción detallada"}, "pedido_id": {"type": "integer", "description": "ID del pedido (opcional)"}}, "required": ["email", "asunto", "descripcion"]}}},
        ]

print('✅ Clase EcoMarketBot definida')

✅ Clase EcoMarketBot definida


## 2. Prueba integrada

In [3]:
# Instanciar el bot
bot = EcoMarketBot(db_path='../data/ecomarket.db')
print('Bot instanciado correctamente')

Bot instanciado correctamente


In [4]:
# Conversación de prueba
preguntas = [
    'Hola, ¿qué productos de limpieza tienen?',
    '¿El detergente tiene alguna promoción?',
    'Quiero ver mis pedidos, soy maria.garcia@email.com',
    '¿Me das el detalle del primer pedido?',
    'Necesito hablar con alguien, tengo un problema con un cargo duplicado',
]

for pregunta in preguntas:
    print(f'\n👤 {pregunta}')
    respuesta = bot.responder(pregunta)
    print(f'🤖 {respuesta}')
    print('-' * 60)


👤 Hola, ¿qué productos de limpieza tienen?
🤖 Tenemos varios productos de limpieza como detergente líquido, jabón de manos, lavavajillas, papel higiénico y suavizante. ¿Necesitas algo específico? 🛍️
------------------------------------------------------------

👤 ¿El detergente tiene alguna promoción?
🤖 No hay promociones vigentes para el detergente líquido. ¿Quieres comprarlo al precio regular de 4.5 unidades? 🛍️
------------------------------------------------------------

👤 Quiero ver mis pedidos, soy maria.garcia@email.com
🤖 ¡Hola Maria! 🙋‍♀️

Tus pedidos son:

- Pedido #21 del 17/06/2026: Confirmado, total 18.0 unidades, se estima entrega para el 19/06/2026.
- Pedido #15 del 23/05/2026: Enviado, total 57.12 unidades, se estima entrega para el 19/06/2026.

¿Necesitas más información o ayuda con algo más? 🛍️
------------------------------------------------------------

👤 ¿Me das el detalle del primer pedido?
🤖 ¡Claro! 📦

El detalle de tu pedido #21 es:

- Leche Entera: 2 unidades x 1

In [5]:
preguntas = [
    'Hola, ¿tienes leche de almendra?',
    '¿puedo pedir lecha a domicilio?',
    'quiero cancelar un pedido que hice'
   
]

for pregunta in preguntas:
    print(f'\n👤 {pregunta}')
    respuesta = bot.responder(pregunta)
    print(f'🤖 {respuesta}')
    print('-' * 60)


👤 Hola, ¿tienes leche de almendra?
🤖 Sí, tenemos leche de almendra. ¿Quieres saber más sobre ella o comprarla? 🥛
------------------------------------------------------------

👤 ¿puedo pedir lecha a domicilio?
🤖 ¡Claro! Puedes pedir leche de almendra a domicilio. ¿Cuál es tu dirección de entrega y cuántas unidades deseas pedir? 🚪

Recuerda que también necesitaré tu email para gestionar tu pedido. 📧
------------------------------------------------------------

👤 quiero cancelar un pedido que hice
🤖 ¿Cuál de los pedidos deseas cancelar? 

1. Pedido #21 del 17/06/2026: Confirmado
2. Pedido #15 del 23/05/2026: Enviado

Ten en cuenta que solo puedo cancelar pedidos que estén en estado "confirmado". Si el pedido ya está "enviado", no puedo cancelarlo. 🚨

¿Cuál es el número del pedido que deseas cancelar? 📝
------------------------------------------------------------


## 3. Chat interactivo (para pruebas en notebook)

In [6]:
# Descomenta para chat interactivo en el notebook
#bot.resetear_conversacion()
#print('💬 Chat con EcoBot (escribe "salir" para terminar)')
#print('=' * 50)
#
# while True:
#     user_input = input('\n👤 Tú: ')
#     if user_input.lower() in ['salir', 'exit', 'quit']:
#         print('\n👋 ¡Hasta luego!')
#         break
#     respuesta = bot.responder(user_input)
#     print(f'\n🤖 EcoBot: {respuesta}')